# Bologna Green Space Accessibility Analysis

**Goal:** Identify which residential buildings in Bologna are within a **300m walking distance** of a publicly accessible park or green space.

**Pipeline overview:**
1. Load city boundary, buildings, parks, and pedestrian network
2. Detect park entrance points (where the pedestrian network intersects park boundaries)
3. Generate 300m walking isochrones from each entrance using GraphHopper
4. Spatial-join buildings against the dissolved isochrones
5. Compute precise building-to-entrance walk distances for matched buildings
6. Export results as GeoJSON, CSV, and SQLite

---

> **GraphHopper must be running before executing the distance cells.**  
> In a separate terminal, navigate to the folder containing `graphhopper-web-11.0.jar`, `config_bologna.yml`, and `Bologna.osm.pbf`, then run:
> ```bash
> rm -rf graph-cache
> java -jar graphhopper-web-11.0.jar server config_bologna.yml
> ```
> Wait for `Started server at HTTP 8989` before continuing.

## 0 · Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import Point, shape
from shapely.strtree import STRtree
from tqdm import tqdm

# ── Coordinate reference systems ──────────────────────────────────────────────
CRS_WGS    = "EPSG:4326"   # WGS-84 geographic (lat/lon) — required by GraphHopper
CRS_METRIC = "EPSG:32632"  # UTM zone 32N — metric distances in metres

# ── Analysis thresholds ────────────────────────────────────────────────────────
WALKING_DISTANCE = 300   # metres — maximum acceptable walk to a park entrance
EUCLID_CUTOFF    = 420   # metres — Euclidean pre-filter; keeps API calls minimal
                         # (straight-line distance is always ≤ walking distance,
                         #  so a ~1.4× buffer safely avoids false negatives)

# ── GraphHopper endpoints ──────────────────────────────────────────────────────
ISO_URL = "http://localhost:8989/isochrone"
GH_URL  = "http://localhost:8989/route"

## 1 · Load Input Data

Four layers are required:
| Layer | Source | CRS after load |
|---|---|---|
| Municipal boundary | `bologna_boundary.gpkg` | UTM 32N |
| Buildings with civic numbers | `bologna_buildings_num.gpkg` | UTM 32N |
| Green spaces (three sub-layers) | `greenery__all_size.gpkg` | UTM 32N |
| Pedestrian network | OSM via `osmnx` | UTM 32N |

In [ ]:
# ── 1a. Municipal boundary ─────────────────────────────────────────────────────
bologna_boundary = gpd.read_file("data/bologna_boundary.gpkg").to_crs(CRS_METRIC)

# WGS-84 version needed by osmnx to download the street network
bologna_wgs84 = bologna_boundary.to_crs(CRS_WGS).geometry.union_all()

In [ ]:
# ── 1b. Buildings ──────────────────────────────────────────────────────────────
buildings = (
    gpd.read_file("data/bologna_buildings_num.gpkg")
    .to_crs(CRS_METRIC)
    .loc[lambda df: df.geometry.notna()]   # drop any null geometries
    .reset_index(drop=True)
)

print(f"{len(buildings):,} buildings loaded")

In [ ]:
# ── 1c. Green spaces ───────────────────────────────────────────────────────────
# Three layers are merged:
#   ar_vrd_all  — municipal gardens / neighbourhood green areas
#   pe_unis_all — green spaces
#   osm_all_public — parks from OpenStreetMap flagged as publicly accessible

parks = (
    pd.concat(
        [gpd.read_file("data/greenery__all_size.gpkg", layer=l)
         for l in ["ar_vrd_all", "pe_unis_all", "osm_all_public"]],
        ignore_index=True,
    )
    .to_crs(CRS_METRIC)
    .reset_index(drop=True)
    .assign(park_id=lambda df: df.index)  # unique integer key used throughout
)

print(f"{len(parks):,} parks (before filtering)")
print(parks["layer_type"].value_counts())

In [ ]:
# ── 1d. Filter ar_vrd layer — remove polygons that enclose buildings ───────────
# Problem: the ar_vrd layer includes courtyards and private gardens that appear
# as green polygons but are not publicly accessible. A practical heuristic:
# if a polygon contains > 6 building footprints it is likely a residential block,
# not a park → exclude it.

building_footprints = (
    gpd.read_file("data/bologna_buildings_final.gpkg")
    .to_crs(CRS_METRIC)
    .loc[lambda df: df.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    .copy()
)

# STRtree for fast spatial lookups
footprint_tree = STRtree(building_footprints.geometry.values)

def buildings_inside(park_geom):
    """Count building footprints that intersect a park polygon."""
    candidates = footprint_tree.query(park_geom, predicate="intersects")
    return len(candidates)

# Apply the count only to ar_vrd (the other layers do not need this filter)
ar_vrd_mask = parks["layer_type"] == "ar_vrd_all"
parks.loc[ar_vrd_mask, "n_buildings_inside"] = (
    parks.loc[ar_vrd_mask, "geometry"].apply(buildings_inside)
)
parks["n_buildings_inside"] = parks["n_buildings_inside"].fillna(0)

parks = parks[
    ~ar_vrd_mask | (parks["n_buildings_inside"] <= 6)
].reset_index(drop=True)

parks["park_id"] = parks.index   # reassign after row removal

print(f"{len(parks):,} parks (after filtering)")
print(parks["layer_type"].value_counts())

In [ ]:
# ── 1e. Pedestrian network ─────────────────────────────────────────────────────
# Download the full walk network for Bologna from OSM and project to UTM 32N.
# The edges GeoDataFrame is used to detect park entrances in the next section.

G = ox.graph_from_polygon(bologna_wgs84, network_type="walk")
G = ox.project_graph(G, to_crs=CRS_METRIC)

_, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index(drop=True)

print(f"{len(edges):,} pedestrian edges")

## 2 · Park Entrance Detection

A *park entrance* is defined as any point where the boundary of a park polygon **intersects a pedestrian edge** in the OSM walk network. These are the candidate start/end points for walking-distance calculations.

Steps:
1. Build an STRtree over all edge geometries (fast bounding-box lookup).
2. For each park, compute `boundary ∩ edges` → collect Point results.
3. Deduplicate entrance points that are closer than 20 m to each other (avoids redundant routing calls where multiple edges share the same crossing).

In [ ]:
# ── Helper: flatten intersection results to a list of Points ──────────────────
def extract_points(geom) -> list:
    """
    Recursively extract individual Point objects from any geometry type.
    A boundary-edge intersection can produce a Point, MultiPoint, or
    GeometryCollection (e.g. a LineString that overlaps a wall segment
    plus a Point where a path crosses the boundary).
    """
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == "Point":
        return [geom]
    if geom.geom_type == "MultiPoint":
        return list(geom.geoms)
    if geom.geom_type == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(extract_points(g))
        return pts
    return []   # LineString / Polygon fragments — not useful as entrance points

In [ ]:
# ── Build spatial index over pedestrian edges ──────────────────────────────────
edges_geom = edges.geometry.values
tree = STRtree(edges_geom)

# ── Compute entrance points ────────────────────────────────────────────────────
rows = []
for _, park in tqdm(parks.iterrows(), total=len(parks), desc="Park entrances"):
    boundary = park.geometry.boundary
    # Query returns indices of edges whose bounding box intersects the boundary
    for idx in tree.query(boundary, predicate="intersects"):
        inter = boundary.intersection(edges_geom[idx])
        for pt in extract_points(inter):
            rows.append({"park_id": park["park_id"], "geometry": pt})

entrances = gpd.GeoDataFrame(rows, crs=CRS_METRIC)
print(f"{len(entrances):,} raw entrance points")

# Parks with zero entrances: likely private gardens not connected to the path
# network. These are excluded from the accessibility analysis.
missing = set(parks["park_id"]) - set(entrances["park_id"])
if missing:
    print(f"{len(missing)} parks with no entrances (likely private): {missing}")

In [ ]:
# ── Deduplicate nearby entrance points ────────────────────────────────────────
# Multiple OSM edges can cross the same section of a park boundary, producing
# near-duplicate points. A greedy nearest-neighbour pass keeps only points
# that are at least 20 m apart within each park.
# Note: the 20 m threshold is not backed by specific literature;
# it is a practical choice to balance coverage and routing efficiency.

def deduplicate_points(group, min_dist=20):
    """Keep only entrance points that are ≥ min_dist metres from each other."""
    kept = []
    for pt in group.geometry:
        if all(pt.distance(k) >= min_dist for k in kept):
            kept.append(pt)
    return gpd.GeoDataFrame({"geometry": kept}, crs=group.crs)

entrances = (
    entrances
    .groupby("park_id", group_keys=True)
    .apply(deduplicate_points)
    .reset_index(level=0)
    .reset_index(drop=True)
    .pipe(gpd.GeoDataFrame, crs=CRS_METRIC)
)

print(f"{len(entrances):,} entrance points after deduplication")

## 3 · Walking Isochrones (GraphHopper)

For each entrance point, request a **300 m walking isochrone** from the local GraphHopper server. The isochrone is a polygon covering all locations reachable within 300 m of walking from that entrance.

Isochrones for the same park are then **dissolved** (unioned) so that a building only needs to fall inside any single entrance's catchment to be considered accessible.

In [ ]:
# ── Pre-compute WGS-84 coordinates for all entrances ─────────────────────────
# GraphHopper expects (lat, lon) — convert once outside the loop.
entrances_park_ids = entrances["park_id"].values
entrances_w_arr    = [(p.y, p.x) for p in entrances.geometry.to_crs(CRS_WGS)]

# ── GraphHopper isochrone call ─────────────────────────────────────────────────
def get_isochrone(lat, lon, distance_m=300):
    """
    Request a distance-based walking isochrone from GraphHopper.
    Returns the GeoJSON polygon feature, or None on failure.
    """
    params = {
        "point":          f"{lat},{lon}",
        "profile":        "foot",
        "distance_limit": distance_m,
        "type":           "json",
    }
    try:
        r = requests.get(ISO_URL, params=params, timeout=10)
        r.raise_for_status()
        return r.json()["polygons"][0]
    except Exception:
        return None


# ── Request isochrones for all entrances ──────────────────────────────────────
iso_rows = []
for i in tqdm(range(len(entrances)), desc="Isochrones"):
    lat, lon = entrances_w_arr[i]
    result   = get_isochrone(lat, lon, distance_m=WALKING_DISTANCE)
    if result:
        iso_rows.append({
            "park_id":      entrances_park_ids[i],
            "entrance_lat": lat,
            "entrance_lon": lon,
            "geometry":     shape(result["geometry"]),
        })

isochrones = gpd.GeoDataFrame(iso_rows, crs=CRS_WGS).to_crs(CRS_METRIC)
print(f"{len(isochrones):,} isochrones computed")

In [ ]:
# ── Dissolve per park ─────────────────────────────────────────────────────────
# Union all entrance isochrones belonging to the same park into one polygon.
# This represents the full 300 m walking catchment of a park, regardless of
# how many entrances it has.

park_reach = isochrones.dissolve(by="park_id").reset_index()
print(f"{len(park_reach):,} parks with at least one isochrone")

## 4 · Spatial Join — Buildings Within 300 m Walk

In [ ]:
# ── Prepare buildings GeoDataFrame ────────────────────────────────────────────
buildings["building_id"] = buildings["indirizzo_completo"]   # unique address key

building_ids    = buildings["building_id"].values
buildings_m_arr = buildings.geometry.values                   # UTM points
buildings_w_arr = [(p.y, p.x) for p in buildings.geometry.to_crs(CRS_WGS)]

buildings_gdf = gpd.GeoDataFrame(
    {"building_id": building_ids},
    geometry=buildings.geometry,
    crs=CRS_METRIC,
)

# ── Spatial join: building point within any park isochrone? ───────────────────
# Left join so that buildings outside all isochrones still appear in the output.
joined = gpd.sjoin(buildings_gdf, park_reach, how="left", predicate="within")

# If isochrones overlap, keep the park with the smallest park_id per building
# (deterministic assignment; does not affect the binary within_300m flag)
joined = (
    joined
    .sort_values("park_id")
    .drop_duplicates(subset="building_id", keep="first")
)

routing_df = joined[["building_id", "park_id", "entrance_lat", "entrance_lon"]].copy()
routing_df["within_300m"] = routing_df["park_id"].notna()

# ── Merge back onto the full buildings table ──────────────────────────────────
buildings_routed = buildings.merge(routing_df, on="building_id", how="left")
buildings_routed["within_300m"] = buildings_routed["within_300m"].fillna(False)

# ── Attach park attributes ────────────────────────────────────────────────────
name_col  = next((c for c in ["nome", "name", "descrizione"] if c in parks.columns), None)
keep_cols = ["park_id", "area_ha", "source", "layer_type"]
if name_col:
    keep_cols.append(name_col)

buildings_routed = buildings_routed.merge(
    parks[keep_cols].copy(), on="park_id", how="left"
)

# ── Rename columns to readable names ─────────────────────────────────────────
rename_dict = {
    "civico":             "street_number",
    "indirizzo_completo": "address",
    "layer_type":         "park_type",
    "area_ha":            "park_area_ha",
    "source":             "park_source",
}
if name_col:
    rename_dict[name_col] = "park_name"
buildings_routed = buildings_routed.rename(columns=rename_dict)

# ── Quick summary ─────────────────────────────────────────────────────────────
n_within = buildings_routed["within_300m"].sum()
pct      = buildings_routed["within_300m"].mean() * 100
print(f"Buildings within {WALKING_DISTANCE}m walk of a park: {n_within:,} ({pct:.1f}%)")

In [ ]:
# ── Intermediate save ─────────────────────────────────────────────────────────
# Park polygons that contribute to the accessibility result are also saved
# so that the GPKG can be opened in QGIS as a self-contained project.

reachable_park_ids = (
    buildings_routed
    .loc[buildings_routed["within_300m"], "park_id"]
    .dropna().unique()
)
parks_within_300m = parks[parks["park_id"].isin(reachable_park_ids)]

buildings_routed.to_file("data/buildings_routed.gpkg", driver="GPKG", layer="buildings")
parks_within_300m.to_file("data/buildings_routed.gpkg", driver="GPKG", layer="parks")
print("Saved buildings_routed.gpkg")

## 5 · Precise Walk Distances (GraphHopper Routing)

For buildings confirmed to be within the 300 m isochrone, compute the **exact routed walking distance** (metres) from the building centroid to its nearest park entrance. This replaces the binary flag with a continuous metric useful for further analysis.

In [ ]:
# ── GraphHopper point-to-point routing call ───────────────────────────────────
def graphhopper(orig_lat, orig_lon, dest_lat, dest_lon):
    """
    Return the shortest walking distance (metres) between two lat/lon points.
    Returns None if the request fails or no path is found.
    """
    params = {
        "point":       [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile":     "foot",
        "calc_points": "false",   # geometry not needed — distance only
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

In [ ]:
# ── Subset: only buildings with a confirmed isochrone match and a known entrance
mask   = buildings_routed["within_300m"] & buildings_routed["entrance_lat"].notna()
subset = buildings_routed[mask].copy()

# Add WGS-84 origin coordinates for each building
subset["orig_lat"] = [p.y for p in subset.geometry.to_crs(CRS_WGS)]
subset["orig_lon"] = [p.x for p in subset.geometry.to_crs(CRS_WGS)]

# ── Route each building to its nearest entrance ───────────────────────────────
walk_distances = []
for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Routing"):
    walk_distances.append(
        graphhopper(
            row["orig_lat"], row["orig_lon"],
            row["entrance_lat"], row["entrance_lon"],
        )
    )

# ── Write distances back to the main table ────────────────────────────────────
# Drop any leftover duplicate columns from previous merge attempts
buildings_routed = buildings_routed.drop(
    columns=["walk_dist_m_x", "walk_dist_m_y"], errors="ignore"
)

buildings_routed["walk_dist_m"] = None
buildings_routed.loc[subset.index, "walk_dist_m"] = walk_distances

# Ensure the object is still a proper GeoDataFrame after index manipulation
buildings_routed = gpd.GeoDataFrame(
    buildings_routed, geometry="geometry", crs=buildings_routed.crs
)

## 6 · Export Results

Three output formats are written to `data/`:

| File | Use |
|---|---|
| `buildings_300m_indicator.geojson` | GIS / web mapping |
| `buildings_300m_indicator.csv` | Tabular analysis / Excel |
| `buildings_300m_indicator.db` | SQL queries (SQLite) |

In [ ]:
# ── Reproject to WGS-84 and add explicit lat/lon columns ─────────────────────
gdf_out = buildings_routed.to_crs(CRS_WGS)
gdf_out["longitude"] = gdf_out.geometry.x
gdf_out["latitude"]  = gdf_out.geometry.y

# ── Select and rename final output columns ────────────────────────────────────
final_cols = {
    "ncivsub":      "street_number",
    "address":      "address",
    "longitude":    "longitude",
    "latitude":     "latitude",
    "quartiere":    "neighbourhood",
    "park_id":      "nearest_park_id",
    "park_type":    "park_type",
    "park_source":  "park_source",
    "park_area_ha": "park_area_ha",
    "entrance_lat": "entrance_lat",
    "entrance_lon": "entrance_lon",
    "walk_dist_m":  "walk_dist_m",
    "within_300m":  "within_300m",
}

keep_cols = list(final_cols.keys()) + ["geometry"]
buildings_out = gpd.GeoDataFrame(
    gdf_out[keep_cols].rename(columns=final_cols),
    geometry="geometry",
    crs=gdf_out.crs,
)

# ── Write outputs ─────────────────────────────────────────────────────────────
buildings_out.to_file("data/buildings_300m_indicator.geojson", driver="GeoJSON")

buildings_out.drop(columns="geometry").to_csv(
    "data/buildings_300m_indicator.csv", index=False
)

conn = sqlite3.connect("data/buildings_300m_indicator.db")
buildings_out.drop(columns="geometry").to_sql(
    "buildings", conn, if_exists="replace", index=False
)
conn.close()

print("Saved: GeoJSON · CSV · SQLite")
print(buildings_out.head())